# Amine Activity Analysis & Thresholding

**Objective:** Analyze BSH enzyme activity with a focus on amines, implementing two thresholding approaches for determining "active" vs "inactive" enzyme-amine pairs.

---

## The Problem

Taurine and glycine are the canonical bile acid conjugates — they're already present in biological samples and can form non-enzymatically. A global intensity threshold would either:
- Miss real activity for low-signal amines (if set too high)
- Include noise as "activity" for high-background amines (if set too low)

## Two Thresholding Approaches

### Approach 1: Per-Amine Control-Based Threshold
For each amine, set threshold = max intensity observed in controls.
- **Pro:** Scientifically defensible; accounts for amine-specific background
- **Pro:** Preserves all amines, including taurine/glycine with genuinely high activity
- **Con:** Taurine/glycine have very few "true positives" above their noise floor

### Approach 2: Exclude Canonical Conjugates
Drop taurine and glycine entirely, focus on non-canonical amines where controls show no/low signal.
- **Pro:** Cleaner signal; these are the biologically interesting amines
- **Pro:** Most non-canonical amines have no control signal → any nonzero = real
- **Con:** Loses information about taurine/glycine specificity

## Configuration & Imports

In [ ]:
# ── Configuration ────────────────────────────────────────────
from pathlib import Path

# Use BSH_model data as source
BSH_MODEL_DIR = Path("../../BSH_model")
HEATMAP_FILE = BSH_MODEL_DIR / "outputs" / "ipsita_heatmap_long.csv"
REACTANTS_FILE = BSH_MODEL_DIR / "data" / "bsh_reactants_SMILES_corrected.xlsx"

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Control enzyme IDs (negative controls and non-BSH enzymes)
CONTROLS = [
    'Negative_ctrl_1', 'Negative_ctrl_2', 'Negative_ctrl_3',
    'Only_substrate_1', 'Only_substrate_2', 'Only_substrate_3',
    'Pencillin_amidase'
]

# Canonical conjugates to exclude in Approach 2
CANONICAL_AMINES = ['taurine', 'glycine']

# For Approach 1: use max or percentile of control
CONTROL_THRESHOLD_PERCENTILE = 100  # 100 = max, 95 = 95th percentile

# Minimum intensity to consider as nonzero (noise floor for approach 2)
EPSILON = 1000

In [ ]:
# ── Imports ──────────────────────────────────────────────────
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import defaultdict

# Optional: seaborn for enhanced plots
try:
    import seaborn as sns
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False
    print("Warning: seaborn not installed. Using matplotlib only.")

# Optional: for UMAP and fingerprints
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("Warning: umap-learn not installed. UMAP visualizations will be skipped.")

try:
    from rdkit import Chem, DataStructs
    from rdkit.Chem import AllChem
    HAS_RDKIT = True
except ImportError:
    HAS_RDKIT = False
    print("Warning: RDKit not installed. Chemical fingerprint analysis will be skipped.")

# Set plot style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('ggplot')
pd.set_option('display.max_columns', 50)

## Load Data & Parse Structure

In [ ]:
# Load heatmap data (long format: Enzyme, ProductName, Intensity)
df = pd.read_csv(HEATMAP_FILE)
print(f"Loaded {len(df):,} rows from heatmap")
print(f"Columns: {df.columns.tolist()}")
df.head()

In [ ]:
def parse_product_name(product_name):
    """
    Parse product name to extract hydroxyl class and amine.
    
    Examples:
        'Di_taurine_29331' -> ('Di', 'taurine', '29331')
        '3a7a12k_taurine_14542' -> ('3a7a12k', 'taurine', '14542')
        'Mono_2_aminophenol_41901' -> ('Mono', '2_aminophenol', '41901')
    """
    parts = str(product_name).split('_')
    
    # First part is hydroxyl class (Mono/Di/Tri or positional like 3a7a12k)
    hydroxyl = parts[0] if parts else None
    
    # Last part is typically rhea_id (numeric)
    rhea_id = None
    if parts and re.match(r'^\d+$', parts[-1]):
        rhea_id = parts[-1]
        parts = parts[:-1]
    
    # Middle parts form the amine name
    if len(parts) > 1:
        amine = '_'.join(parts[1:])
    else:
        amine = None
    
    # Normalize amine name
    if amine:
        amine = amine.lower().replace('m+nh4', '').replace('m+h2o', '').strip('_')
        # Handle special cases
        if 'diaminopropionic' in amine:
            amine = '2,3_diaminopropionic acid'
        elif 'methoxytyramine' in amine:
            amine = '3_methoxytyramine'
        elif 'aminophenol' in amine:
            amine = '2_aminophenol'
        elif 'aminobutyric' in amine or amine == 'gaba':
            amine = 'gaba'
    
    return hydroxyl, amine, rhea_id

# Parse all product names
parsed = df['ProductName'].apply(parse_product_name)
df['Hydroxyl'] = parsed.apply(lambda x: x[0])
df['Amine'] = parsed.apply(lambda x: x[1])
df['RheaID'] = parsed.apply(lambda x: x[2])

print(f"\nUnique hydroxyl classes: {df['Hydroxyl'].nunique()}")
print(df['Hydroxyl'].value_counts())

In [ ]:
# Check unique amines
print(f"Unique amines: {df['Amine'].nunique()}")
print("\nAmine value counts:")
print(df.groupby('Amine')['ProductName'].nunique().sort_values(ascending=False))

In [ ]:
# Separate controls from BSH enzymes
df['IsControl'] = df['Enzyme'].isin(CONTROLS)

df_controls = df[df['IsControl']].copy()
df_bsh = df[~df['IsControl']].copy()

print(f"Control measurements: {len(df_controls):,} ({df_controls['Enzyme'].nunique()} enzymes)")
print(f"BSH measurements: {len(df_bsh):,} ({df_bsh['Enzyme'].nunique()} enzymes)")
print(f"\nControl enzymes: {df_controls['Enzyme'].unique().tolist()}")

In [ ]:
# Summary statistics
print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(f"Total enzymes: {df['Enzyme'].nunique()} ({df_bsh['Enzyme'].nunique()} BSH + {df_controls['Enzyme'].nunique()} controls)")
print(f"Total products: {df['ProductName'].nunique()}")
print(f"Total amines: {df['Amine'].nunique()}")
print(f"Total measurements: {len(df):,}")
print(f"\nIntensity statistics:")
print(f"  Zero measurements: {(df['Intensity'] == 0).sum():,} ({100*(df['Intensity'] == 0).mean():.1f}%)")
print(f"  Non-zero measurements: {(df['Intensity'] > 0).sum():,} ({100*(df['Intensity'] > 0).mean():.1f}%)")
print(f"  Max intensity: {df['Intensity'].max():,.0f}")
print(f"  Mean (non-zero): {df.loc[df['Intensity'] > 0, 'Intensity'].mean():,.0f}")
print(f"  Median (non-zero): {df.loc[df['Intensity'] > 0, 'Intensity'].median():,.0f}")

## Intensity Distribution Visualization

In [ ]:
# Histogram of all intensities (log scale)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All intensities
nonzero = df.loc[df['Intensity'] > 0, 'Intensity']
axes[0].hist(np.log10(nonzero + 1), bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('log10(Intensity + 1)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Distribution of Non-Zero Intensities (n={len(nonzero):,})')
axes[0].axvline(np.log10(EPSILON), color='red', linestyle='--', label=f'Epsilon={EPSILON}')
axes[0].legend()

# BSH vs Controls
bsh_nonzero = df_bsh.loc[df_bsh['Intensity'] > 0, 'Intensity']
ctrl_nonzero = df_controls.loc[df_controls['Intensity'] > 0, 'Intensity']

axes[1].hist(np.log10(bsh_nonzero + 1), bins=50, alpha=0.6, label=f'BSH (n={len(bsh_nonzero):,})', edgecolor='black')
axes[1].hist(np.log10(ctrl_nonzero + 1), bins=50, alpha=0.6, label=f'Controls (n={len(ctrl_nonzero):,})', edgecolor='black')
axes[1].set_xlabel('log10(Intensity + 1)')
axes[1].set_ylabel('Count')
axes[1].set_title('BSH vs Control Intensity Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'intensity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-amine intensity distributions (boxplot)
# Aggregate by amine for visualization
amine_stats = df.groupby('Amine').agg({
    'Intensity': ['mean', 'median', 'max', 'sum'],
    'ProductName': 'nunique'
}).round(0)
amine_stats.columns = ['mean', 'median', 'max', 'sum', 'n_products']
amine_stats = amine_stats.sort_values('max', ascending=False)

# Get top amines by max intensity
top_amines = amine_stats.head(20).index.tolist()

fig, ax = plt.subplots(figsize=(14, 8))

# Filter to top amines and non-zero intensities
plot_data = df[df['Amine'].isin(top_amines) & (df['Intensity'] > 0)].copy()
plot_data['log_intensity'] = np.log10(plot_data['Intensity'] + 1)

# Order by median intensity
order = plot_data.groupby('Amine')['log_intensity'].median().sort_values(ascending=False).index.tolist()

if HAS_SEABORN:
    sns.boxplot(data=plot_data, x='Amine', y='log_intensity', hue='IsControl', 
                order=order, ax=ax, palette={True: 'red', False: 'steelblue'})
    ax.legend(title='Is Control')
else:
    # Fallback to matplotlib boxplot
    bsh_data = [plot_data[(plot_data['Amine'] == a) & (~plot_data['IsControl'])]['log_intensity'].values for a in order]
    ctrl_data = [plot_data[(plot_data['Amine'] == a) & (plot_data['IsControl'])]['log_intensity'].values for a in order]
    positions = np.arange(len(order))
    bp1 = ax.boxplot(bsh_data, positions=positions-0.2, widths=0.35, patch_artist=True)
    bp2 = ax.boxplot(ctrl_data, positions=positions+0.2, widths=0.35, patch_artist=True)
    for patch in bp1['boxes']:
        patch.set_facecolor('steelblue')
    for patch in bp2['boxes']:
        patch.set_facecolor('red')
    ax.set_xticks(positions)
    ax.set_xticklabels(order)
    ax.legend([bp1['boxes'][0], bp2['boxes'][0]], ['BSH', 'Control'])

ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_ylabel('log10(Intensity + 1)')
ax.set_title('Intensity Distribution by Amine (Top 20 by Max Intensity)')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'amine_intensity_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## Compute Per-Amine Control Thresholds

In [ ]:
def compute_amine_thresholds(df_controls, percentile=100):
    """
    Compute per-amine thresholds based on control intensities.
    
    Args:
        df_controls: DataFrame with control measurements
        percentile: 100 for max, or lower value (e.g., 95) for percentile
    
    Returns:
        DataFrame with amine thresholds and statistics
    """
    thresholds = []
    
    for amine in df_controls['Amine'].unique():
        amine_data = df_controls[df_controls['Amine'] == amine]['Intensity']
        
        if percentile == 100:
            threshold = amine_data.max()
        else:
            threshold = np.percentile(amine_data, percentile)
        
        thresholds.append({
            'Amine': amine,
            'control_max': amine_data.max(),
            'control_95pct': np.percentile(amine_data, 95),
            'control_mean': amine_data.mean(),
            'control_median': amine_data.median(),
            'n_control_measurements': len(amine_data),
            'n_control_nonzero': (amine_data > 0).sum(),
            'threshold': threshold
        })
    
    return pd.DataFrame(thresholds)

# Compute thresholds
thresholds_df = compute_amine_thresholds(df_controls, CONTROL_THRESHOLD_PERCENTILE)
thresholds_df = thresholds_df.sort_values('control_max', ascending=False)

print(f"Computed thresholds for {len(thresholds_df)} amines")
print("\nTop 10 amines by control signal:")
thresholds_df.head(10)

In [ ]:
# Amines with NO control signal (threshold = 0)
no_signal_amines = thresholds_df[thresholds_df['control_max'] == 0]['Amine'].tolist()
print(f"\nAmines with NO control signal ({len(no_signal_amines)}/{len(thresholds_df)}):")
print(no_signal_amines)

In [ ]:
# Visualize control thresholds
fig, ax = plt.subplots(figsize=(14, 6))

# Sort by threshold
plot_thresh = thresholds_df.sort_values('control_max', ascending=True)

colors = ['red' if a.lower() in [c.lower() for c in CANONICAL_AMINES] else 'steelblue' 
          for a in plot_thresh['Amine']]

bars = ax.barh(plot_thresh['Amine'], plot_thresh['control_max'], color=colors, alpha=0.7)
ax.set_xlabel('Control Max Intensity (Threshold)')
ax.set_ylabel('Amine')
ax.set_title('Per-Amine Control Thresholds (Red = Canonical Conjugates)')

# Add text annotations for high values
for i, (amine, val) in enumerate(zip(plot_thresh['Amine'], plot_thresh['control_max'])):
    if val > 1e6:
        ax.text(val + 1e5, i, f'{val/1e6:.1f}M', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'control_thresholds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Add BSH statistics to thresholds
bsh_stats = df_bsh.groupby('Amine').agg({
    'Intensity': ['count', lambda x: (x > 0).sum(), 'max', 'mean']
}).reset_index()
bsh_stats.columns = ['Amine', 'n_bsh_total', 'n_bsh_nonzero', 'bsh_max', 'bsh_mean']

thresholds_df = thresholds_df.merge(bsh_stats, on='Amine', how='left')

# Compute how many BSH measurements are above threshold
def count_above_threshold(row):
    amine = row['Amine']
    threshold = row['threshold']
    amine_bsh = df_bsh[df_bsh['Amine'] == amine]
    return (amine_bsh['Intensity'] > threshold).sum()

thresholds_df['n_bsh_above_threshold'] = thresholds_df.apply(count_above_threshold, axis=1)

# Save thresholds
thresholds_df.to_csv(OUTPUT_DIR / 'amine_activity_thresholds.csv', index=False)
print(f"Saved thresholds to {OUTPUT_DIR / 'amine_activity_thresholds.csv'}")
thresholds_df.head(10)

## Approach 1: Per-Amine Control-Based Activity

In [ ]:
# Create threshold lookup
amine_threshold_map = thresholds_df.set_index('Amine')['threshold'].to_dict()

# Apply thresholds to BSH data
df_bsh['threshold'] = df_bsh['Amine'].map(amine_threshold_map)
df_bsh['active_approach1'] = df_bsh['Intensity'] > df_bsh['threshold']

print("Approach 1: Per-Amine Control-Based Thresholding")
print("=" * 60)
print(f"Total BSH measurements: {len(df_bsh):,}")
print(f"Active (above threshold): {df_bsh['active_approach1'].sum():,} ({100*df_bsh['active_approach1'].mean():.1f}%)")
print(f"Inactive: {(~df_bsh['active_approach1']).sum():,} ({100*(~df_bsh['active_approach1']).mean():.1f}%)")

In [ ]:
# Create activity matrix for Approach 1 (enzyme x amine)
# Aggregate across products for same enzyme-amine pair (take max)
activity1 = df_bsh.groupby(['Enzyme', 'Amine']).agg({
    'Intensity': 'max',
    'active_approach1': 'any',  # Active if ANY product for this enzyme-amine is active
    'threshold': 'first'
}).reset_index()

# Pivot to wide format
activity_matrix1 = activity1.pivot(index='Enzyme', columns='Amine', values='active_approach1')
activity_matrix1 = activity_matrix1.fillna(False).astype(int)

print(f"\nActivity Matrix (Approach 1): {activity_matrix1.shape}")
print(f"  Enzymes: {activity_matrix1.shape[0]}")
print(f"  Amines: {activity_matrix1.shape[1]}")

# Summary per amine
amine_activity_summary1 = activity_matrix1.sum().sort_values(ascending=False)
print(f"\nActive enzymes per amine (Approach 1):")
print(amine_activity_summary1)

In [ ]:
# Summary per enzyme
enzyme_activity_summary1 = activity_matrix1.sum(axis=1).sort_values(ascending=False)
print(f"\nAmines per enzyme (Approach 1) - Top 20:")
print(enzyme_activity_summary1.head(20))
print(f"\nEnzymes with 0 active amines: {(enzyme_activity_summary1 == 0).sum()}")

In [ ]:
# Save Approach 1 activity matrix
activity_matrix1.to_csv(OUTPUT_DIR / 'activity_matrix_approach1.csv')
print(f"Saved: {OUTPUT_DIR / 'activity_matrix_approach1.csv'}")

## Approach 2: Exclude Canonical Conjugates

In [ ]:
# Filter out canonical amines (taurine, glycine)
canonical_lower = [c.lower() for c in CANONICAL_AMINES]
df_bsh['is_canonical'] = df_bsh['Amine'].str.lower().isin(canonical_lower)

df_noncanonical = df_bsh[~df_bsh['is_canonical']].copy()

print("Approach 2: Exclude Canonical Conjugates")
print("=" * 60)
print(f"Original measurements: {len(df_bsh):,}")
print(f"After removing taurine/glycine: {len(df_noncanonical):,}")
print(f"Removed: {len(df_bsh) - len(df_noncanonical):,} measurements")

In [ ]:
# For non-canonical amines: active if intensity > epsilon (or > 0 for amines with no control signal)
df_noncanonical['active_approach2'] = df_noncanonical['Intensity'] > EPSILON

print(f"\nUsing epsilon = {EPSILON} as activity threshold")
print(f"Active (above epsilon): {df_noncanonical['active_approach2'].sum():,} ({100*df_noncanonical['active_approach2'].mean():.1f}%)")
print(f"Inactive: {(~df_noncanonical['active_approach2']).sum():,}")

In [ ]:
# Create activity matrix for Approach 2
activity2 = df_noncanonical.groupby(['Enzyme', 'Amine']).agg({
    'Intensity': 'max',
    'active_approach2': 'any'
}).reset_index()

activity_matrix2 = activity2.pivot(index='Enzyme', columns='Amine', values='active_approach2')
activity_matrix2 = activity_matrix2.fillna(False).astype(int)

print(f"\nActivity Matrix (Approach 2): {activity_matrix2.shape}")
print(f"  Enzymes: {activity_matrix2.shape[0]}")
print(f"  Amines: {activity_matrix2.shape[1]} (excluding taurine, glycine)")

# Summary per amine
amine_activity_summary2 = activity_matrix2.sum().sort_values(ascending=False)
print(f"\nActive enzymes per amine (Approach 2):")
print(amine_activity_summary2)

In [ ]:
# Summary per enzyme
enzyme_activity_summary2 = activity_matrix2.sum(axis=1).sort_values(ascending=False)
print(f"\nAmines per enzyme (Approach 2) - Top 20:")
print(enzyme_activity_summary2.head(20))
print(f"\nEnzymes with 0 active amines: {(enzyme_activity_summary2 == 0).sum()}")

In [ ]:
# Save Approach 2 activity matrix
activity_matrix2.to_csv(OUTPUT_DIR / 'activity_matrix_approach2.csv')
print(f"Saved: {OUTPUT_DIR / 'activity_matrix_approach2.csv'}")

## Compare Approaches

In [ ]:
# Compare the two approaches
print("COMPARISON: Approach 1 vs Approach 2")
print("=" * 60)

# Get common amines (those in both approaches)
common_amines = set(activity_matrix1.columns) & set(activity_matrix2.columns)
print(f"Amines in Approach 1: {len(activity_matrix1.columns)}")
print(f"Amines in Approach 2: {len(activity_matrix2.columns)} (excludes {CANONICAL_AMINES})")
print(f"Common amines: {len(common_amines)}")

# Total active pairs
total_active_1 = activity_matrix1.sum().sum()
total_active_2 = activity_matrix2.sum().sum()
print(f"\nTotal active (enzyme, amine) pairs:")
print(f"  Approach 1: {total_active_1:,}")
print(f"  Approach 2: {total_active_2:,}")

In [ ]:
# Per-amine comparison (for common amines)
comparison_df = pd.DataFrame({
    'amine': list(common_amines),
    'n_active_approach1': [activity_matrix1[a].sum() for a in common_amines],
    'n_active_approach2': [activity_matrix2[a].sum() for a in common_amines]
})
comparison_df['difference'] = comparison_df['n_active_approach2'] - comparison_df['n_active_approach1']
comparison_df = comparison_df.sort_values('difference', ascending=False)

print("\nPer-amine comparison (sorted by difference):")
comparison_df

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot: approach 1 vs approach 2 active counts per amine
axes[0].scatter(comparison_df['n_active_approach1'], comparison_df['n_active_approach2'], alpha=0.7)
axes[0].plot([0, comparison_df['n_active_approach1'].max()], 
             [0, comparison_df['n_active_approach1'].max()], 'r--', label='y=x')
axes[0].set_xlabel('Active Enzymes (Approach 1)')
axes[0].set_ylabel('Active Enzymes (Approach 2)')
axes[0].set_title('Per-Amine Activity: Approach 1 vs 2')
axes[0].legend()

# Add amine labels for outliers
for _, row in comparison_df.iterrows():
    if abs(row['difference']) > 10:
        axes[0].annotate(row['amine'], (row['n_active_approach1'], row['n_active_approach2']),
                        fontsize=8, alpha=0.7)

# Bar plot: difference by amine
plot_comp = comparison_df.head(20)  # Top 20 by absolute difference
colors = ['green' if d > 0 else 'red' for d in plot_comp['difference']]
axes[1].barh(plot_comp['amine'], plot_comp['difference'], color=colors, alpha=0.7)
axes[1].axvline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('Difference (Approach 2 - Approach 1)')
axes[1].set_title('Activity Difference by Amine')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'approach_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Enzyme Activity Profiles

In [ ]:
# Heatmap of enzyme x amine activity (Approach 1)
fig, ax = plt.subplots(figsize=(16, 12))

# Sort enzymes by total activity
enzyme_order = activity_matrix1.sum(axis=1).sort_values(ascending=False).index
amine_order = activity_matrix1.sum().sort_values(ascending=False).index

# Only show top 50 enzymes for visibility
top_enzymes = enzyme_order[:50]
heatmap_data = activity_matrix1.loc[top_enzymes, amine_order]

if HAS_SEABORN:
    sns.heatmap(heatmap_data, cmap='YlOrRd', cbar_kws={'label': 'Active'}, ax=ax)
else:
    im = ax.imshow(heatmap_data.values, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(amine_order)))
    ax.set_xticklabels(amine_order, rotation=90)
    ax.set_yticks(range(len(top_enzymes)))
    ax.set_yticklabels(top_enzymes)
    plt.colorbar(im, ax=ax, label='Active')

ax.set_title('Enzyme x Amine Activity Matrix (Approach 1, Top 50 Enzymes)')
ax.set_xlabel('Amine')
ax.set_ylabel('Enzyme')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'activity_heatmap_approach1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap of enzyme x amine activity (Approach 2)
fig, ax = plt.subplots(figsize=(16, 12))

# Sort enzymes by total activity
enzyme_order2 = activity_matrix2.sum(axis=1).sort_values(ascending=False).index
amine_order2 = activity_matrix2.sum().sort_values(ascending=False).index

top_enzymes2 = enzyme_order2[:50]
heatmap_data2 = activity_matrix2.loc[top_enzymes2, amine_order2]

if HAS_SEABORN:
    sns.heatmap(heatmap_data2, cmap='YlOrRd', cbar_kws={'label': 'Active'}, ax=ax)
else:
    im = ax.imshow(heatmap_data2.values, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(amine_order2)))
    ax.set_xticklabels(amine_order2, rotation=90)
    ax.set_yticks(range(len(top_enzymes2)))
    ax.set_yticklabels(top_enzymes2)
    plt.colorbar(im, ax=ax, label='Active')

ax.set_title('Enzyme x Amine Activity Matrix (Approach 2, Top 50 Enzymes)')
ax.set_xlabel('Amine')
ax.set_ylabel('Enzyme')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'activity_heatmap_approach2.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cluster enzymes by activity profile (Approach 2)
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

# Filter to enzymes with at least 1 active amine
active_enzymes = activity_matrix2[activity_matrix2.sum(axis=1) > 0]

if len(active_enzymes) > 5 and HAS_SEABORN:
    # Compute distances and linkage
    distances = pdist(active_enzymes.values, metric='jaccard')
    Z = linkage(distances, method='ward')
    
    # Clustermap
    g = sns.clustermap(active_enzymes, cmap='YlOrRd', figsize=(14, 10),
                       row_linkage=Z, col_cluster=True,
                       cbar_kws={'label': 'Active'})
    g.fig.suptitle('Clustered Enzyme Activity Profiles (Approach 2)', y=1.02)
    
    plt.savefig(OUTPUT_DIR / 'activity_clustermap_approach2.png', dpi=150, bbox_inches='tight')
    plt.show()
elif len(active_enzymes) > 5:
    # Fallback: just show dendrogram
    distances = pdist(active_enzymes.values, metric='jaccard')
    Z = linkage(distances, method='ward')
    
    fig, ax = plt.subplots(figsize=(14, 6))
    dendrogram(Z, labels=active_enzymes.index.tolist(), ax=ax, leaf_rotation=90)
    ax.set_title('Enzyme Clustering by Activity Profile (Approach 2)')
    ax.set_ylabel('Distance')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'activity_dendrogram_approach2.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Not enough active enzymes for clustering")

## Amine Chemical Space vs Activity

In [ ]:
# Load amine SMILES from reactants file
if HAS_RDKIT and REACTANTS_FILE.exists():
    reactants = pd.read_excel(REACTANTS_FILE)
    print(f"Loaded {len(reactants)} reactants")
    print(f"Columns: {reactants.columns.tolist()}")
    
    # Filter to amines
    if 'Compount Type' in reactants.columns:
        type_col = 'Compount Type'
    elif 'Compound Type' in reactants.columns:
        type_col = 'Compound Type'
    else:
        type_col = None
    
    if type_col:
        amines_df = reactants[reactants[type_col].str.lower().str.contains('amine', na=False)].copy()
        print(f"Found {len(amines_df)} amine entries")
        display(amines_df[['Compound_Name', 'SMILES']].head(10))
    else:
        amines_df = reactants.copy()
else:
    print("RDKit not available or reactants file not found. Skipping chemical analysis.")
    amines_df = None

In [ ]:
if HAS_RDKIT and HAS_UMAP and amines_df is not None and len(amines_df) > 0:
    # Compute Morgan fingerprints
    def get_fingerprint(smiles, radius=2, nBits=1024):
        try:
            mol = Chem.MolFromSmiles(str(smiles))
            if mol is None:
                return None
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nBits)
            arr = np.zeros((nBits,), dtype=np.int8)
            DataStructs.ConvertToNumpyArray(fp, arr)
            return arr
        except:
            return None
    
    # Compute fingerprints for amines
    amines_df['fingerprint'] = amines_df['SMILES'].apply(get_fingerprint)
    valid_amines = amines_df[amines_df['fingerprint'].notna()].copy()
    
    if len(valid_amines) >= 5:
        # Stack fingerprints
        fps = np.stack(valid_amines['fingerprint'].values)
        
        # UMAP embedding
        reducer = umap.UMAP(n_neighbors=min(15, len(valid_amines)-1), 
                           min_dist=0.1, random_state=42)
        embedding = reducer.fit_transform(fps)
        
        valid_amines['umap_x'] = embedding[:, 0]
        valid_amines['umap_y'] = embedding[:, 1]
        
        # Normalize amine names for matching
        valid_amines['amine_normalized'] = valid_amines['Compound_Name'].str.lower().str.replace(' ', '_')
        
        # Match with activity data
        activity_by_amine = activity_matrix2.sum().to_dict()
        valid_amines['n_active_enzymes'] = valid_amines['amine_normalized'].map(
            lambda x: activity_by_amine.get(x, 0)
        )
        
        # Plot UMAP colored by activity
        fig, ax = plt.subplots(figsize=(10, 8))
        scatter = ax.scatter(valid_amines['umap_x'], valid_amines['umap_y'],
                           c=valid_amines['n_active_enzymes'], 
                           cmap='viridis', s=100, alpha=0.7)
        plt.colorbar(scatter, label='# Active Enzymes')
        
        # Add labels for high-activity amines
        for _, row in valid_amines.iterrows():
            if row['n_active_enzymes'] > 5:
                ax.annotate(row['Compound_Name'], (row['umap_x'], row['umap_y']),
                           fontsize=8, alpha=0.8)
        
        ax.set_xlabel('UMAP 1')
        ax.set_ylabel('UMAP 2')
        ax.set_title('Amine Chemical Space (UMAP of Morgan Fingerprints)\nColored by # Active Enzymes')
        
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'amine_umap_activity.png', dpi=150, bbox_inches='tight')
        plt.show()
    else:
        print(f"Not enough valid amines for UMAP ({len(valid_amines)})")
else:
    print("Skipping UMAP visualization (missing dependencies or data)")

## Export for Model Training

In [ ]:
# Create comprehensive long-format output with both activity labels
# Start with BSH data
output_df = df_bsh[['Enzyme', 'ProductName', 'Amine', 'Hydroxyl', 'Intensity', 'threshold']].copy()
output_df['active_approach1'] = df_bsh['active_approach1']

# Add approach 2 activity (only for non-canonical amines)
output_df['active_approach2'] = False  # Default
noncanonical_mask = ~output_df['Amine'].str.lower().isin(canonical_lower)
output_df.loc[noncanonical_mask, 'active_approach2'] = output_df.loc[noncanonical_mask, 'Intensity'] > EPSILON

# Add flag for canonical amines
output_df['is_canonical'] = output_df['Amine'].str.lower().isin(canonical_lower)

print(f"Output DataFrame: {len(output_df):,} rows")
print(f"Columns: {output_df.columns.tolist()}")
output_df.head(10)

In [ ]:
# Save long-format output
output_df.to_csv(OUTPUT_DIR / 'enzyme_amine_activity.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'enzyme_amine_activity.csv'}")

In [ ]:
# Create amine summary statistics
amine_summary = thresholds_df.copy()

# Add approach 1 activity counts
approach1_counts = activity_matrix1.sum().reset_index()
approach1_counts.columns = ['Amine', 'n_enzymes_active_approach1']
amine_summary = amine_summary.merge(approach1_counts, on='Amine', how='left')

# Add approach 2 activity counts (only for non-canonical)
approach2_counts = activity_matrix2.sum().reset_index()
approach2_counts.columns = ['Amine', 'n_enzymes_active_approach2']
amine_summary = amine_summary.merge(approach2_counts, on='Amine', how='left')

# Flag canonical amines
amine_summary['is_canonical'] = amine_summary['Amine'].str.lower().isin(canonical_lower)

# Reorder columns
cols = ['Amine', 'is_canonical', 'control_max', 'control_95pct', 'control_mean', 
        'n_control_nonzero', 'threshold', 'n_bsh_total', 'n_bsh_nonzero', 
        'n_bsh_above_threshold', 'n_enzymes_active_approach1', 'n_enzymes_active_approach2']
cols = [c for c in cols if c in amine_summary.columns]
amine_summary = amine_summary[cols].sort_values('n_enzymes_active_approach1', ascending=False)

# Save
amine_summary.to_csv(OUTPUT_DIR / 'amine_activity_summary.csv', index=False)
print(f"Saved: {OUTPUT_DIR / 'amine_activity_summary.csv'}")
amine_summary

In [ ]:
# Final summary
print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)
print(f"\nFiles generated:")
print(f"  1. {OUTPUT_DIR / 'amine_activity_thresholds.csv'} - Per-amine threshold values")
print(f"  2. {OUTPUT_DIR / 'activity_matrix_approach1.csv'} - Binary activity (control-based threshold)")
print(f"  3. {OUTPUT_DIR / 'activity_matrix_approach2.csv'} - Binary activity (exclude canonical)")
print(f"  4. {OUTPUT_DIR / 'amine_activity_summary.csv'} - Per-amine statistics")
print(f"  5. {OUTPUT_DIR / 'enzyme_amine_activity.csv'} - Long-format with both approaches")

print(f"\n--- Approach 1: Per-Amine Control-Based Thresholding ---")
print(f"  Total amines: {len(activity_matrix1.columns)}")
print(f"  Total enzymes: {len(activity_matrix1)}")
print(f"  Total active (enzyme, amine) pairs: {activity_matrix1.sum().sum():,}")
print(f"  Enzymes with at least 1 active amine: {(activity_matrix1.sum(axis=1) > 0).sum()}")

print(f"\n--- Approach 2: Exclude Canonical Conjugates ---")
print(f"  Total amines: {len(activity_matrix2.columns)} (excludes {CANONICAL_AMINES})")
print(f"  Total enzymes: {len(activity_matrix2)}")
print(f"  Total active (enzyme, amine) pairs: {activity_matrix2.sum().sum():,}")
print(f"  Enzymes with at least 1 active amine: {(activity_matrix2.sum(axis=1) > 0).sum()}")

## Recommendations

### Which Approach to Use?

**Approach 2 (Exclude Canonical)** is recommended for initial model training because:
1. Cleaner signal-to-noise for non-canonical amines
2. Most non-canonical amines have zero/minimal control signal
3. Biologically more interesting - these are the amines where BSH specificity matters

**Approach 1** can be used for:
- Validating the model on canonical conjugates
- Understanding complete substrate specificity profiles
- Cases where taurine/glycine activity differences matter

### Next Steps
1. Combine activity labels with enzyme embeddings (from ProtT5 or conservation-filtered embeddings)
2. Train classifier on Approach 2 data
3. Validate on held-out enzymes
4. Optionally, use Approach 1 as additional validation set